# Multi-Task Hard-Sharing Multi-Branch CNN

Targets:

- `gesture_action` primary head
- `orientation` auxiliary head
- `phase` auxiliary head

Model:

- shared multi-branch CNN backbone
- attention / BiGRU / none fusion
- three task heads
- uncertainty-weighted task losses
- multi-domain sensor extraction
- handedness and upside-down correction inside the pipeline
- optional Kalman / EKF-style smoothing and dead-reckoning features


In [1]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.metrics import f1_score, classification_report

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
except Exception:
    BayesSearchCV = None
    Categorical = None
    Integer = None
    Real = None

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/hardsharer/')
sys.path.append('/kaggle/input/cmi-competition-code')

import data_utils
import utils_multitask_hardsharing as utils


2026-05-21 20:11:42.094516: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779394302.274773      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779394302.326935      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779394302.758035      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779394302.758073      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779394302.758077      24 computation_placer.cc:177] computation placer alr

In [2]:
# ============================================================
# Config
# ============================================================

search_mode = "bayesian"      # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 10
n_jobs = 1

use_subject_holdout = False
holdout_size = 0.5
train_size = 0.2      # used only when use_subject_holdout=False
chosen_orientation = None

pipe_name = "sequence_builder"
corrector_name = "sensor_corrector"
classifier_name = "classifier"

primary_target = "gesture_action"
orientation_target = "orientation"

# Reuse the existing third "phase" head as a position/location head.
# No utils rewrite needed.
position_target = "gesture_position"
phase_target = position_target

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")


In [3]:
# ============================================================
# Load data
# ============================================================

data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

print(raw_train_df.shape)
print(raw_test_df.shape)
print(train_demo_df.shape)
print(test_demo_df.shape)


Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
(574945, 341)
(107, 336)
(81, 8)
(2, 8)


In [4]:
# ============================================================
# Base dataframe + helper targets
# ============================================================

train_df = raw_train_df.set_index("row_id").copy(deep=True)

train_df.loc[:, "gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df.loc[:, "gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df.loc[:, "is_target"] = train_df["sequence_type"].eq("Target").astype(int)
train_df.loc[:, "phase_target"] = train_df["phase"]
train_df.loc[:, "orientation_action"] = train_df["orientation"].astype(str) + "||" + train_df["gesture_action"].astype(str)

print("all sequences:", train_df["sequence_id"].nunique())
print("target sequences:", train_df.loc[train_df["sequence_type"].eq("Target"), "sequence_id"].nunique())
print("gesture_action classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_action"].dropna().unique()))
print("orientation classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "orientation"].dropna().unique()))
print("gesture_position classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_position"].dropna().unique()))
print("full gesture classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture"].dropna().unique()))


all sequences: 8151
target sequences: 5113
gesture_action classes: ['pinch skin', 'pull hair', 'pull hairline', 'scratch']
orientation classes: ['Lie on Back', 'Lie on Side - Non Dominant', 'Seated Lean Non Dom - FACE DOWN', 'Seated Straight']
gesture_position classes: ['Above ear', 'Cheek', 'Eyebrow', 'Eyelash', 'Forehead', 'Neck']
full gesture classes: ['Above ear - pull hair', 'Cheek - pinch skin', 'Eyebrow - pull hair', 'Eyelash - pull hair', 'Forehead - pull hairline', 'Forehead - scratch', 'Neck - pinch skin', 'Neck - scratch']


In [5]:
# ============================================================
# Split: subject holdout OR old train_size logic
# ============================================================

if use_subject_holdout:
    seq_meta = (
        train_df
        .drop_duplicates("sequence_id")
        [["sequence_id", "subject", "sequence_type", "gesture", "gesture_action", "orientation", "phase"]]
        .reset_index(drop=True)
    )

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=holdout_size,
        random_state=random_state,
    )

    train_seq_idx, holdout_seq_idx = next(
        splitter.split(
            seq_meta,
            y=seq_meta["gesture"],
            groups=seq_meta["subject"],
        )
    )

    train_seq_ids = seq_meta.loc[train_seq_idx, "sequence_id"]
    holdout_seq_ids = seq_meta.loc[holdout_seq_idx, "sequence_id"]

    train_sample_df = train_df.loc[train_df["sequence_id"].isin(train_seq_ids)].copy()
    test_sample_df = train_df.loc[train_df["sequence_id"].isin(holdout_seq_ids)].copy()

else:
    if train_size is None:
        rows = (
            (train_demo_df["adult_child"] == 1)
            & (train_demo_df["sex"] == 1)
            & (train_demo_df["handedness"] == 1)
        )

        ideal_subject_ids = (
            train_demo_df.loc[rows]
            .sort_values("elbow_to_wrist_cm", ascending=False)["subject"]
            .to_list()
        )

        train_sample_df = train_df.loc[
            train_df["subject"].isin(ideal_subject_ids)
            & train_df["sequence_type"].eq("Target")
        ].copy()

        test_sample_df = None

    elif train_size == 0:
        some_sequences = train_df["sequence_id"].unique()[10:20]

        train_sample_df = train_df.loc[
            train_df["sequence_id"].isin(some_sequences)
        ].copy()

        test_sample_df = None

    else:
        target_df = train_df.loc[train_df["sequence_type"].eq("Target")].copy()

        train_sample_df, test_sample_df = data_utils.sample_balanced_split(
            target_df,
            train_pct=train_size,
            test_pct=0.2,
            random_state=random_state,
        )

        train_sample_df = train_sample_df.copy()
        test_sample_df = test_sample_df.copy()

if chosen_orientation is not None:
    train_sample_df = train_sample_df.loc[train_sample_df["orientation"].isin(chosen_orientation)].copy()
    if test_sample_df is not None:
        test_sample_df = test_sample_df.loc[test_sample_df["orientation"].isin(chosen_orientation)].copy()

target_only_train_df = train_sample_df.loc[train_sample_df["sequence_type"].eq("Target")].copy()

target_only_holdout_df = None
if test_sample_df is not None:
    target_only_holdout_df = test_sample_df.loc[test_sample_df["sequence_type"].eq("Target")].copy()

print("use_subject_holdout:", use_subject_holdout)
print("train sequences:", train_sample_df["sequence_id"].nunique())
print("target-only train sequences:", target_only_train_df["sequence_id"].nunique())

if test_sample_df is not None:
    print("holdout/test sequences:", test_sample_df["sequence_id"].nunique())
    print("target-only holdout sequences:", target_only_holdout_df["sequence_id"].nunique())
    print("train subjects:", train_sample_df["subject"].nunique())
    print("holdout subjects:", test_sample_df["subject"].nunique())
    print("subject overlap:", len(set(train_sample_df["subject"]) & set(test_sample_df["subject"])))


Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%
use_subject_holdout: False
train sequences: 648
target-only train sequences: 648
holdout/test sequences: 648
target-only holdout sequences: 648
train subjects: 81
holdout subjects: 81
subject overlap: 81


In [6]:
# ============================================================
# Mapping check: orientation + gesture_action -> original gesture
# ============================================================

target_map_df = (
    target_only_train_df
    .drop_duplicates(["orientation", "gesture_action", "gesture"])
    [["orientation", "gesture_action", "gesture"]]
    .copy()
)

mapping_check = (
    target_map_df
    .groupby(["orientation", "gesture_action"])["gesture"]
    .nunique()
    .reset_index(name="n_gesture")
    .sort_values("n_gesture", ascending=False)
)

print(mapping_check.head(20))
print("max mappings per orientation/action:", mapping_check["n_gesture"].max())


                        orientation gesture_action  n_gesture
1                       Lie on Back      pull hair          3
5        Lie on Side - Non Dominant      pull hair          3
13                  Seated Straight      pull hair          3
9   Seated Lean Non Dom - FACE DOWN      pull hair          3
7        Lie on Side - Non Dominant        scratch          2
4        Lie on Side - Non Dominant     pinch skin          2
3                       Lie on Back        scratch          2
0                       Lie on Back     pinch skin          2
15                  Seated Straight        scratch          2
12                  Seated Straight     pinch skin          2
11  Seated Lean Non Dom - FACE DOWN        scratch          2
8   Seated Lean Non Dom - FACE DOWN     pinch skin          2
6        Lie on Side - Non Dominant  pull hairline          1
2                       Lie on Back  pull hairline          1
10  Seated Lean Non Dom - FACE DOWN  pull hairline          1
14      

In [7]:
# ============================================================
# Pipeline
# ============================================================

pipeline = Pipeline([
    (
        corrector_name,
        utils.SensorOrientationCorrector(
            demo_df=train_demo_df,
            apply_handedness=True,
            apply_upside_down=True,
        ),
    ),
    (
        pipe_name,
        utils.AdvancedMultiDomainSequenceExtractor(
            acc_modes=("raw", "velocity", "displacement", "jerk"),
            rotation_modes=("quaternion", "rot6d", "angular_velocity"),
            sampling_rate=10,
            compute_dt=True,
            interp_mode="linear",
            standardize="mean_std",
            tof_mode="pooled_diff",
            tof_fill_mode="nan_interpolate",
            thm_mode="centered_diff",
            use_acc_magnitude=True,
            use_linear_acc_magnitude=True,
            linear_acc_mode="baseline",
            motion_filter_mode=None,
            use_dead_reckoning=False,
        ),
    ),
    (
        classifier_name,
        utils.KerasMultiTaskMultiBranchClassifier(
            gesture_target=primary_target,
            orientation_target=orientation_target,
            phase_target=phase_target,
            maxlen=160,
            branch_filters={"acc": "64-128-256-256", "rot": "64-64", "tof": "64", "thm": "16"},
            branch_kernel_sizes={"acc": "5-5-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
            branch_pool_sizes={"acc": "2-2-none-none", "rot": "none-none", "tof": "none", "thm": "none"},
            fusion_mode="attention",
            attention_heads=4,
            gru_units=170,
            dense_units="64",
            dropout=0.45,
            spatial_dropout=0.1,
            learning_rate=2e-4,
            batch_size=16,
            epochs=140,
            patience=15,
            uncertainty_weighting=False,
            fixed_gesture_weight=1.0,
            fixed_orientation_weight=0.3,
            fixed_phase_weight=1.0,
            verbose=0,
            random_state=random_state,
        ),
    ),
])


In [8]:
# ============================================================
# Search space
# ============================================================

if search_mode == "bayesian":
    param_space = {
        # ============================================================
        # Sequence builder / feature extraction
        # ============================================================
        f"{pipe_name}__acc_modes": Categorical([
            #"raw",
            #"raw|velocity|displacement",
            "smoothed|velocity|displacement|jerk",
        ]),

        f"{pipe_name}__rotation_modes": Categorical([
            #"quaternion|rot6d",
            #"quaternion|rot6d|angular_velocity",
            "quaternion|euler|rot6d|angular_velocity",
            #"rot6d|angular_velocity",
        ]),

        f"{pipe_name}__sampling_rate": Categorical([10]),
        f"{pipe_name}__interp_mode": Categorical(["ffill"]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),

        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),

        f"{pipe_name}__tof_mode": Categorical([
            #"sensor_stats",
            "pooled_stats",
            #"pooled_diff",
        ]),
        f"{pipe_name}__tof_fill_mode": Categorical([
            #"zero",
            "far_255",
            #"far_500"
        ]),
        f"{pipe_name}__thm_mode": Categorical([
           # "raw",
            "centered",
           # "diff",
           # "centered_diff",
        ]),

        f"{pipe_name}__motion_filter_mode": Categorical([
           # None,
           # "kalman",
            "extended_kalman",
        ]),
        f"{pipe_name}__kalman_process_noise": Real(
            1e-5,
            1e-1,
            prior="log-uniform",
        ),
        f"{pipe_name}__kalman_measurement_noise": Real(
            1e-4,
            2.0,
            prior="log-uniform",
        ),
        f"{pipe_name}__use_dead_reckoning": Categorical([ True]),

        f"{pipe_name}__clip_value": Categorical([50.0]),
        f"{pipe_name}__window_size": Integer(2, 100),
        f"{pipe_name}__smooth_alpha": Categorical([None]),

        # ============================================================
        # Multi-task multi-branch classifier
        # IMPORTANT:
        # Keep these as raw lists of dicts.
        # prepare_multitask_param_space converts them safely for Bayes.
        # ============================================================
        f"{classifier_name}__maxlen": Integer(2, 200),

        f"{classifier_name}__branch_filters": [
            #{"acc": "16-32", "rot": "16", "tof": "16", "thm": "8"},
            #{"acc": "32-64", "rot": "32", "tof": "32", "thm": "8"},
           # {"acc": "64-128-256", "rot": "64-64", "tof": "64", "thm": "16"},
           # {"acc": "64-128-256-256", "rot": "64-64", "tof": "64", "thm": "16"},
            {"acc": "128-256-256-256", "rot": "128-128", "tof": "128", "thm": "32"},
            {"acc": "128-256-512", "rot": "128-256", "tof": "64", "thm": "16"}
        ],

        f"{classifier_name}__branch_kernel_sizes": [
            {"acc": "3-3", "rot": "3-3", "tof": "3", "thm": "3"},
            {"acc": "5-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
            {"acc": "3-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
            {"acc": "5-5-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
            {"acc": "7-5-3", "rot": "3-3", "tof": "3", "thm": "3"},
        ],

        f"{classifier_name}__branch_pool_sizes": [
            {"acc": "none", "rot": "none", "tof": "none", "thm": "none"},
            #{"acc": "2", "rot": "2", "tof": "2", "thm": "2"},
            #{"acc": "2-2", "rot": "2-2", "tof": "2-2", "thm": "2-2"},
            #{"acc": "2-2-none-none", "rot": "none-none", "tof": "none", "thm": "none"},
        ],

        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__fusion_mode": Categorical(["attention"]),
        f"{classifier_name}__attention_heads": Integer(2, 15),
        f"{classifier_name}__gru_units": Integer(64, 400),

        f"{classifier_name}__dense_units": Categorical([
            "none",
            "16",
            "32",
            "64",
            "128",
            "64-32",
            "128-64",
        ]),

        # f"{classifier_name}__dropout": Real(0.0, 0.6),
        # f"{classifier_name}__spatial_dropout": Real(0.0, 0.4),
        # f"{classifier_name}__learning_rate": Real(
        #     1e-6,
        #     5e-4,
        #     prior="log-uniform",
        # ),
        # f"{classifier_name}__batch_size": Categorical([8, 16, 32]),
        f"{classifier_name}__epochs": Categorical([120]),
        # f"{classifier_name}__patience": Categorical([10, 15, 20]),

        # ============================================================
        # Multi-task learning
        # ============================================================
        f"{classifier_name}__uncertainty_weighting": Categorical([True]),
        f"{classifier_name}__fixed_gesture_weight": Categorical([1.0]),
        f"{classifier_name}__fixed_orientation_weight": Categorical([1.0]),
        f"{classifier_name}__fixed_phase_weight": Categorical([1.0]),

        # ============================================================
        # Augmentation — only supported params
        # ============================================================
        f"{classifier_name}__use_mixup": Categorical([False]),
        # f"{classifier_name}__mixup_alpha": Real(0.1, 0.8),
        # f"{classifier_name}__mixup_size": Real(0.5, 1.5),
        # f"{classifier_name}__mixup_prob": Real(0.5, 1.0),

        # f"{classifier_name}__use_gaussian_noise": Categorical([False, True]),
        # f"{classifier_name}__noise_std": Real(
        #     0.001,
        #     0.05,
        #     prior="log-uniform",
        # ),

        # f"{classifier_name}__use_magnitude_scaling": Categorical([False, True]),
        # f"{classifier_name}__magnitude_scale_min": Real(0.8, 0.95),
        # f"{classifier_name}__magnitude_scale_max": Real(1.05, 1.2),

        # f"{classifier_name}__use_time_mask": Categorical([False, True]),
        # f"{classifier_name}__time_mask_ratio": Real(0.05, 0.3),

        # f"{classifier_name}__use_time_shift": Categorical([False, True]),
        # f"{classifier_name}__max_shift_pct": Real(0.1, 0.4),

        # f"{classifier_name}__use_channel_dropout": Categorical([False, True]),
        # f"{classifier_name}__channel_dropout_prob": Real(0.0, 0.3),

        # f"{classifier_name}__use_modality_dropout": Categorical([False, True]),
        # f"{classifier_name}__drop_acc_prob": Real(0.0, 0.3),
        # f"{classifier_name}__drop_rot_prob": Real(0.0, 0.3),
        # f"{classifier_name}__drop_tof_prob": Real(0.0, 0.5),
        # f"{classifier_name}__drop_thm_prob": Real(0.0, 0.5),
    }

    param_space = utils.prepare_multitask_param_space(
        param_space,
        search_mode,
        Categorical=Categorical,
    )


elif search_mode == "grid":
    param_space = {
        # ============================================================
        # Sequence builder / feature extraction
        # ============================================================
        f"{pipe_name}__acc_modes": [
            ("raw", "velocity", "displacement"),
            ("smoothed", "velocity", "displacement", "jerk"),
        ],

        f"{pipe_name}__rotation_modes": [
            ("quaternion", "rot6d", "angular_velocity"),
            ("quaternion", "euler", "rot6d", "angular_velocity"),
        ],

        f"{pipe_name}__sampling_rate": [10],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__standardize": ["mean_std"],
        f"{pipe_name}__linear_acc_mode": ["baseline"],

        f"{pipe_name}__use_acc_magnitude": [True],
        f"{pipe_name}__use_linear_acc_magnitude": [True],

        f"{pipe_name}__tof_mode": ["pooled_diff"],
        f"{pipe_name}__tof_fill_mode": ["nan_interpolate"],
        f"{pipe_name}__thm_mode": ["centered_diff"],

        f"{pipe_name}__motion_filter_mode": [None, "kalman"],
        f"{pipe_name}__kalman_process_noise": [1e-3],
        f"{pipe_name}__kalman_measurement_noise": [1e-1],
        f"{pipe_name}__use_dead_reckoning": [False, True],

        f"{pipe_name}__clip_value": [None],
        f"{pipe_name}__window_size": [12],
        f"{pipe_name}__smooth_alpha": [0.8],

        # ============================================================
        # Multi-task multi-branch classifier
        # ============================================================
        f"{classifier_name}__maxlen": [160],

        f"{classifier_name}__branch_filters": [
            {"acc": "64-128-256-256", "rot": "64-64", "tof": "64", "thm": "16"},
        ],

        f"{classifier_name}__branch_kernel_sizes": [
            {"acc": "5-5-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
        ],

        f"{classifier_name}__branch_pool_sizes": [
            {"acc": "2-2-none-none", "rot": "none-none", "tof": "none", "thm": "none"},
        ],

        f"{classifier_name}__use_batch_norm": [True],
        f"{classifier_name}__fusion_mode": ["attention"],
        f"{classifier_name}__attention_heads": [4],
        f"{classifier_name}__gru_units": [170],

        f"{classifier_name}__dense_units": ["64"],
        f"{classifier_name}__dropout": [0.45],
        f"{classifier_name}__spatial_dropout": [0.1],
        f"{classifier_name}__learning_rate": [2e-4],
        f"{classifier_name}__batch_size": [16],
        f"{classifier_name}__epochs": [120],
        f"{classifier_name}__patience": [15],

        # ============================================================
        # Multi-task learning
        # ============================================================
        f"{classifier_name}__uncertainty_weighting": [False],
        f"{classifier_name}__fixed_gesture_weight": [1.0],
        f"{classifier_name}__fixed_orientation_weight": [0.3],
        f"{classifier_name}__fixed_phase_weight": [1.0],

        # ============================================================
        # Augmentation — only supported params
        # ============================================================
        f"{classifier_name}__use_mixup": [False, True],
        f"{classifier_name}__mixup_alpha": [0.4, 0.5],
        f"{classifier_name}__mixup_size": [1.0],
        f"{classifier_name}__mixup_prob": [1.0],

        f"{classifier_name}__use_gaussian_noise": [False],
        f"{classifier_name}__noise_std": [0.01],

        f"{classifier_name}__use_magnitude_scaling": [False],
        f"{classifier_name}__magnitude_scale_min": [0.9],
        f"{classifier_name}__magnitude_scale_max": [1.1],

        f"{classifier_name}__use_time_mask": [False],
        f"{classifier_name}__time_mask_ratio": [0.1],

        f"{classifier_name}__use_time_shift": [False],
        f"{classifier_name}__max_shift_pct": [0.25],

        f"{classifier_name}__use_channel_dropout": [False],
        f"{classifier_name}__channel_dropout_prob": [0.1],

        f"{classifier_name}__use_modality_dropout": [False],
        f"{classifier_name}__drop_acc_prob": [0.0],
        f"{classifier_name}__drop_rot_prob": [0.0],
        f"{classifier_name}__drop_tof_prob": [0.0],
        f"{classifier_name}__drop_thm_prob": [0.0],
    }

In [9]:
# ============================================================
# CV search
# ============================================================

cv = GroupKFold(n_splits=n_splits)

groups = target_only_train_df["subject"]
X_train = target_only_train_df.copy()
y_train = target_only_train_df[["sequence_id", primary_target, orientation_target, phase_target]].copy()

if search_mode == "bayesian":
    if BayesSearchCV is None:
        raise ImportError("skopt is not installed. Use search_mode='grid' or install scikit-optimize.")

    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        cv=cv,
        scoring=None,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        cv=cv,
        scoring=None,
        n_jobs=n_jobs,
        verbose=4,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

search.fit(X_train, y_train, groups=groups)

print("best gesture macro F1:", search.best_score_)
print("best params:")
print(search.best_params_)


Fitting 2 folds for each of 1 candidates, totalling 2 fits


I0000 00:00:1779394393.669006      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779394393.675074      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1779394401.694220      70 service.cc:152] XLA service 0x7fb26400a9f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779394401.694271      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779394401.694278      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779394403.089706      70 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779394413.019580      70 device_compiler.h:188] Compiled clust

[CV 1/2] END classifier__attention_heads=7, classifier__branch_filters={"acc": "128-256-512", "rot": "128-256", "thm": "16", "tof": "64"}, classifier__branch_kernel_sizes={"acc": "7-5-3", "rot": "3-3", "thm": "3", "tof": "3"}, classifier__branch_pool_sizes={"acc": "none", "rot": "none", "thm": "none", "tof": "none"}, classifier__dense_units=128, classifier__epochs=120, classifier__fixed_gesture_weight=1.0, classifier__fixed_orientation_weight=1.0, classifier__fixed_phase_weight=1.0, classifier__fusion_mode=attention, classifier__gru_units=249, classifier__maxlen=27, classifier__uncertainty_weighting=True, classifier__use_batch_norm=True, classifier__use_mixup=False, sequence_builder__acc_modes=smoothed|velocity|displacement|jerk, sequence_builder__clip_value=50.0, sequence_builder__interp_mode=ffill, sequence_builder__kalman_measurement_noise=0.00015007926915787495, sequence_builder__kalman_process_noise=0.08691422834366752, sequence_builder__linear_acc_mode=baseline, sequence_builder_

2026-05-21 20:26:50.614280: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng18{k11=0} for conv %cudnn-conv-bw-filter.6 = (f32[128,128,1,3]{3,2,1,0}, u8[0]{0}) custom-call(f32[16,128,1,168]{3,2,1,0} %bitcast.36107, f32[16,128,1,168]{3,2,1,0} %bitcast.35323), window={size=1x3 pad=0_0x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardFilter", metadata={op_type="Conv2DBackpropFilter" op_name="gradient_tape/uncertainty_weighted_mtl_model_1/conv1d_5_1/convolution/Conv2DBackpropFilter" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-05-21 20:26:50.751160: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation too

[CV 1/2] END classifier__attention_heads=13, classifier__branch_filters={"acc": "128-256-256-256", "rot": "128-128", "thm": "32", "tof": "128"}, classifier__branch_kernel_sizes={"acc": "3-5-5", "rot": "3-3", "thm": "3", "tof": "3"}, classifier__branch_pool_sizes={"acc": "none", "rot": "none", "thm": "none", "tof": "none"}, classifier__dense_units=64, classifier__epochs=120, classifier__fixed_gesture_weight=1.0, classifier__fixed_orientation_weight=1.0, classifier__fixed_phase_weight=1.0, classifier__fusion_mode=attention, classifier__gru_units=179, classifier__maxlen=168, classifier__uncertainty_weighting=True, classifier__use_batch_norm=True, classifier__use_mixup=False, sequence_builder__acc_modes=smoothed|velocity|displacement|jerk, sequence_builder__clip_value=50.0, sequence_builder__interp_mode=ffill, sequence_builder__kalman_measurement_noise=0.13261354317645616, sequence_builder__kalman_process_noise=0.0002055148115052581, sequence_builder__linear_acc_mode=baseline, sequence_bui

In [10]:
# ============================================================
# Save CV results
# ============================================================

cv_results_df = pd.DataFrame(search.cv_results_)
cv_path = results_dir / f"{search_mode}_multitask_hardsharing_cv_results_{timestamp}.csv"
cv_results_df.to_csv(cv_path, index=False)

best_params_path = results_dir / f"{search_mode}_multitask_hardsharing_best_params_{timestamp}.json"
with open(best_params_path, "w") as f:
    json.dump(search.best_params_, f, indent=2, default=str)

print(cv_path)
print(best_params_path)


results/bayesian_multitask_hardsharing_cv_results_20260521_2012.csv
results/bayesian_multitask_hardsharing_best_params_20260521_2012.json


In [11]:
# ============================================================
# Holdout evaluation
# ============================================================

best_model = search.best_estimator_

if target_only_holdout_df is not None and not target_only_holdout_df.empty:
    X_holdout = target_only_holdout_df.copy()

    holdout_seq = (
        X_holdout
        .drop_duplicates("sequence_id")
        [["sequence_id", primary_target, orientation_target, position_target, "gesture"]]
        .reset_index(drop=True)
    )

    X_holdout_features = best_model[:-1].transform(X_holdout)

    pred_df = best_model.named_steps[classifier_name].predict_all(
        X_holdout_features
    )

    # The utils class still calls the third head "phase_pred".
    # In this notebook, phase_pred is actually gesture_position_pred.
    pred_df = pred_df.rename(
        columns={
            "phase_pred": "gesture_position_pred"
        }
    )

    holdout_pred = pd.concat(
        [
            holdout_seq.reset_index(drop=True),
            pred_df.reset_index(drop=True),
        ],
        axis=1,
    )

    holdout_pred["reconstructed_gesture_pred"] = (
        holdout_pred["gesture_position_pred"].astype(str)
        + " - "
        + holdout_pred["gesture_action_pred"].astype(str)
    )

    gesture_action_f1 = f1_score(
        holdout_pred[primary_target],
        holdout_pred["gesture_action_pred"],
        average="macro",
    )

    gesture_position_f1 = f1_score(
        holdout_pred[position_target],
        holdout_pred["gesture_position_pred"],
        average="macro",
    )

    orientation_f1 = f1_score(
        holdout_pred[orientation_target],
        holdout_pred["orientation_pred"],
        average="macro",
    )

    reconstructed_gesture_f1 = f1_score(
        holdout_pred["gesture"],
        holdout_pred["reconstructed_gesture_pred"],
        average="macro",
    )

    print("holdout gesture_action macro F1:", round(gesture_action_f1, 4))
    print("holdout gesture_position macro F1:", round(gesture_position_f1, 4))
    print("holdout orientation macro F1:", round(orientation_f1, 4))
    print("holdout reconstructed gesture macro F1:", round(reconstructed_gesture_f1, 4))

    print("\nGesture action report")
    print(classification_report(
        holdout_pred[primary_target],
        holdout_pred["gesture_action_pred"],
    ))

    print("\nGesture position report")
    print(classification_report(
        holdout_pred[position_target],
        holdout_pred["gesture_position_pred"],
    ))

    print("\nOrientation report")
    print(classification_report(
        holdout_pred[orientation_target],
        holdout_pred["orientation_pred"],
    ))

    print("\nReconstructed full gesture report")
    print(classification_report(
        holdout_pred["gesture"],
        holdout_pred["reconstructed_gesture_pred"],
    ))

    holdout_path = results_dir / f"multitask_position_holdout_predictions_{timestamp}.csv"
    summary_path = results_dir / f"multitask_position_holdout_summary_{timestamp}.csv"

    holdout_pred.to_csv(holdout_path, index=False)

    pd.DataFrame([
        {
            "search_mode": search_mode,
            "best_cv_gesture_action_macro_f1": search.best_score_,
            "holdout_gesture_action_macro_f1": gesture_action_f1,
            "holdout_gesture_position_macro_f1": gesture_position_f1,
            "holdout_orientation_macro_f1": orientation_f1,
            "holdout_reconstructed_gesture_macro_f1": reconstructed_gesture_f1,
            "n_train_sequences": target_only_train_df["sequence_id"].nunique(),
            "n_holdout_sequences": target_only_holdout_df["sequence_id"].nunique(),
        }
    ]).to_csv(summary_path, index=False)

    print(holdout_path)
    print(summary_path)

else:
    print("No target-only holdout dataframe available. Skipping holdout evaluation.")

holdout gesture_action macro F1: 0.4386
holdout gesture_position macro F1: 0.4243
holdout orientation macro F1: 0.9584
holdout reconstructed gesture macro F1: 0.1981

Gesture action report
               precision    recall  f1-score   support

   pinch skin       0.56      0.46      0.50       162
    pull hair       0.56      0.69      0.62       243
pull hairline       0.30      0.21      0.25        81
      scratch       0.39      0.38      0.39       162

     accuracy                           0.50       648
    macro avg       0.45      0.44      0.44       648
 weighted avg       0.48      0.50      0.49       648


Gesture position report
              precision    recall  f1-score   support

   Above ear       0.52      0.47      0.49        81
       Cheek       0.33      0.19      0.24        81
     Eyebrow       0.22      0.23      0.23        81
     Eyelash       0.28      0.21      0.24        81
    Forehead       0.69      0.81      0.74       162
        Neck      

In [12]:
# ============================================================
# Inspect task setup
# ============================================================

clf = search.best_estimator_.named_steps[classifier_name]

print("gesture target:", clf.gesture_target)
print("orientation target:", clf.orientation_target)
print("third head target:", clf.phase_target)

print("uncertainty weighting:", clf.uncertainty_weighting)
print("fixed gesture weight:", clf.fixed_gesture_weight)
print("fixed orientation weight:", clf.fixed_orientation_weight)
print("fixed position/phase weight:", clf.fixed_phase_weight)

if hasattr(clf, "gesture_classes_"):
    print("\ngesture_action classes:")
    print(clf.gesture_classes_)

if hasattr(clf, "orientation_classes_"):
    print("\norientation classes:")
    print(clf.orientation_classes_)

if hasattr(clf, "phase_classes_"):
    print("\ngesture_position classes:")
    print(clf.phase_classes_)

gesture target: gesture_action
orientation target: orientation
third head target: gesture_position
uncertainty weighting: True
fixed gesture weight: 1.0
fixed orientation weight: 1.0
fixed position/phase weight: 1.0

gesture_action classes:
['pinch skin' 'pull hair' 'pull hairline' 'scratch']

orientation classes:
['Lie on Back' 'Lie on Side - Non Dominant'
 'Seated Lean Non Dom - FACE DOWN' 'Seated Straight']

gesture_position classes:
['Above ear' 'Cheek' 'Eyebrow' 'Eyelash' 'Forehead' 'Neck']
